# Loading the data

In [36]:
import pandas as pd

train_data = pd.read_csv("data/train.csv")

test_data = pd.read_csv("data/test.csv")

solution = pd.read_csv("data/solution.csv")

# train_data.columns.tolist()
# test_data


train_only_cols = set(train_data.columns) - set(test_data.columns)
print(train_only_cols)

test_only_cols = set(test_data.columns)  - set(train_data.columns)
print(test_only_cols)

train_data.columns

{'player_rating'}
set()


Index(['Id', 'player_id', 'player_name', 'age', 'nationality', 'team',
       'jersey_number', 'position', 'height_cm', 'weight_kg', 'preferred_foot',
       'club_name', 'market_value_eur', 'match_id', 'match_date', 'stadium',
       'city', 'opponent_team', 'tournament_stage', 'match_result',
       'goals_team', 'goals_opponent', 'minutes_played', 'goals', 'assists',
       'shots', 'shots_on_target', 'expected_goals_xg', 'expected_assists_xa',
       'key_passes', 'successful_passes', 'total_passes', 'pass_accuracy',
       'dribbles_attempted', 'successful_dribbles', 'crosses',
       'successful_crosses', 'tackles', 'interceptions', 'clearances',
       'blocks', 'aerial_duels_won', 'aerial_duels_lost', 'recoveries',
       'defensive_actions', 'fouls_committed', 'fouls_suffered',
       'yellow_cards', 'red_cards', 'offsides', 'saves', 'save_percentage',
       'punches', 'clean_sheet', 'goals_conceded', 'penalty_saves',
       'distance_covered_km', 'sprint_distance_km', 'top_s

As can be seen from the data, there are many irrelevant features for evaluating the player's rating. So we must choose from these features.

# Preprocessing the data

In [ ]:
# columns_to_drop = [
#     'Id', 'player_id', 'player_name', 'match_id', 'match_date', 
#     'stadium', 'city', 'jersey_number', 'nationality', 'team', 
#     'club_name', 'opponent_team', 'tournament_stage', 
#     'market_value_eur',
#     'performance_score', 'player_of_match_awards' # Dropping to avoid leakage
# ]

columns_to_drop = ['Id', 'player_id', 'player_name', 'nationality', 'team',
       'jersey_number',
       'club_name', 'market_value_eur', 'match_id', 'match_date', 'stadium',
       'city', 'opponent_team', 'tournament_stage',
       'performance_score', 'offensive_contribution', 'defensive_contribution',
       'possession_impact', 'pressure_resistance', 'creativity_score',
       'consistency_score', 'clutch_performance_score',
       'total_goals_tournament', 'total_assists_tournament',
       'total_minutes_tournament']

X_train = train_data.drop(columns=columns_to_drop + ['player_rating'])
Y_train = train_data['player_rating']
y_true = solution['player_rating']

X_test = test_data.drop(columns=columns_to_drop)

# One hot encoding for string features
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
# X_train

# Training the model

In [38]:
from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor(max_depth=5, random_state=42)

tree_model.fit(X_train, Y_train)

DecisionTreeRegressor(max_depth=5, random_state=42)

# Testing the model

In [39]:
predections = tree_model.predict(X_test)



## Error using $R^2$

In [40]:
model_score = tree_model.score(X_test, y_true)
print(f"Model Score: {model_score:.4f}")

Model Score: 0.9712


## Error using MSE

In [41]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_true, predections)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 0.2802755792529148


# Observations

For some reason, the variance of the dataset is mostly explained by the `minutes_played` feature!

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': tree_model.feature_importances_
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("Top 10 Most Important Features:")
print(importance_df.head(10))

Top 10 Most Important Features:
                Feature  Importance
5        minutes_played    0.995608
14         total_passes    0.002832
6                 goals    0.000744
47     position_Forward    0.000300
39  distance_covered_km    0.000282
8                 shots    0.000181
40   sprint_distance_km    0.000041
26           recoveries    0.000013
34      save_percentage    0.000000
41        top_speed_kmh    0.000000


# Saving submission